In [1]:
from luxai_s3.params import EnvParams

from luxai_s3.wrappers import LuxAIS3GymEnv
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import CallbackList, CheckpointCallback
import os
import sys
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize, SubprocVecEnv, DummyVecEnv
import torch
# Add the project root directory to the Python path
sys.path.append(os.path.abspath(".."))
from wrappers.base_wrapper import SB3LuxEnvBase
from common.helper import EnhancedTensorboardCallback, make_env, custom_env_check, linear_schedule

## Check if the environment is compatible with SB3

In [2]:
# Create the base environment
base_env = LuxAIS3GymEnv()
# Create environment parameters
env_params = EnvParams(map_type=0, max_steps_in_match=100)

# Apply our wrapper with explicit player_id and opponent strategy
wrapped_env = SB3LuxEnvBase(base_env, player_id='player_0', opponent_strategy='random')

# Create the environment
env = make_env(env_params, wrapped_env, seed=367)  # Using a fixed seed for reproducibility

# Print observation space shape
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

check_env(env)
print("Environment check passed successfully!")

Observation space: Dict('env_cfg_map_height': Box(0, 24, (1,), int32), 'env_cfg_map_width': Box(0, 24, (1,), int32), 'env_cfg_max_steps_in_match': Box(0, 100, (1,), int32), 'env_cfg_unit_move_cost': Box(0, 100, (1,), int32), 'env_cfg_unit_sap_cost': Box(0, 100, (1,), int32), 'env_cfg_unit_sap_range': Box(0, 100, (1,), int32), 'map_features_energy': Box(-1, 20, (24, 24), int8), 'map_features_tile_type': Box(-1, 2, (24, 24), int8), 'match_steps': Box(0, 100, (1,), int32), 'relic_nodes': Box(-1, 23, (6, 2), int32), 'relic_nodes_mask': Box(0, 1, (6,), int8), 'remainingOverageTime': Box(0, 1000, (1,), int32), 'sensor_mask': Box(0, 1, (24, 24), int8), 'steps': Box(0, 100, (1,), int32), 'team_points': Box(0, 1000, (2,), int32), 'team_wins': Box(0, 1000, (2,), int32), 'units_energy': Box(0, 400, (2, 16, 1), int32), 'units_mask': Box(0, 1, (2, 16), int8), 'units_position': Box(-1, 575, (2, 16, 2), int32))
Action space: MultiDiscrete([6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6])


/opt/anaconda3/envs/inf367/lib/python3.11/site-packages/stable_baselines3/common/env_checker.py:272: UserWarning: Your observation map_features_energy has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(
/opt/anaconda3/envs/inf367/lib/python3.11/site-packages/stable_baselines3/common/env_checker.py:272: UserWarning: Your observation map_features_tile_type has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(
/opt/anaconda3/envs/inf367/lib/python3.11/site-packages/stable_baselines3/common/env_checker.py:272: UserWarning: Your observation relic_nodes has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a c

Environment check passed successfully!


#### Custom enviorment check 🤖

In [3]:
# custom_env_check(env, 2)

### Train the base PPO model

In [4]:
models_dir = "../ppo_lux_model_base/"
SAVE_FREQUENCY = 1000

envs = SubprocVecEnv(
    [lambda: make_env(env_params, wrapped_env, seed=i) for i in range(8)]
)
envs = VecNormalize(envs, norm_obs=True, norm_reward=True)

# Create callbacks
enchared_tb_callback = EnhancedTensorboardCallback()
checkpoint_callback = CheckpointCallback(
    save_freq=SAVE_FREQUENCY, save_path=models_dir, name_prefix="ppo_lux_model_base"
)

# Combine callbacks
callbacks = CallbackList([enchared_tb_callback, checkpoint_callback])

mps_device = None
if not torch.backends.mps.is_available():
    if not torch.backends.mps.is_built():
        print(
            "MPS not available because the current PyTorch install was not "
            "built with MPS enabled."
        )
    else:
        print(
            "MPS not available because the current MacOS version is not 12.3+ "
            "and/or you do not have an MPS-enabled device on this machine."
        )
else:
    mps_device = torch.device("mps")

    print("MPS available and enabled.")

# Create PPO model
model = PPO(
    "MultiInputPolicy",
    envs,
    verbose=1,
    learning_rate=linear_schedule(5e-4, 1e-4),
    n_steps=2048,
    batch_size=128,
    n_epochs=5,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    vf_coef=0.25,
    ent_coef=3.0,
    clip_range_vf=0.2,
    tensorboard_log="./ppo_lux_tensorboard/",
    device=mps_device,
    policy_kwargs=dict(net_arch=[512, 256, 128], activation_fn=torch.nn.ReLU),
)

# Train the model with callbacks
model.learn(total_timesteps=1000000, callback=callbacks)

# Save final model
model.save("ppo_lux_model_base")

MPS available and enabled.
Using mps device
Logging to ./ppo_lux_tensorboard/PPO_1
-----------------------------------------
| lux/                       |          |
|    energy_collected        | 0        |
|    final_reward            | 16.6     |
|    map_coverage            | 6.94     |
|    new_tiles_revealed      | 3        |
|    point_reward            | 0        |
|    points_earned           | 0        |
|    relic_control_streak    | 0        |
|    relic_point_tiles_found | 0        |
|    rule_reward             | 16.6     |
|    sap_actions_taken       | 3        |
|    sap_reward              | 0        |
|    total_energy            | 158      |
|    visited_tiles_count     | 11       |
|    win_rate                | 0        |
| time/                      |          |
|    fps                     | 108      |
|    iterations              | 1        |
|    time_elapsed            | 150      |
|    total_timesteps         | 16384    |
-----------------------------------